# Урок 1.3. Выбор между облачными API и локальными моделями

**Модуль 1 «Введение в LLM» · Курс «LLM для Python-разработчиков»**

В уроках 1.1–1.2 мы разобрались, *как* LLM устроена и *как* с ней разговаривать. Теперь — инженерный вопрос номер один, который встанет в любом реальном проекте: **где эта модель будет работать?** У вас два пути: платить облачному провайдеру за API или запускать модель на своём железе. Это решение влияет на бюджет, приватность данных, скорость ответов и объём вашей ops-работы — и его часто принимают наобум.

## 🎯 Цели урока

После этого урока вы сможете:

- объяснить, чем архитектурно отличаются «модель как сервис» и «модель как процесс» — и что из этого следует;
- **посчитать стоимость** облачной модели под конкретную нагрузку и найти точку безубыточности против своего железа;
- оценить, **какая локальная модель поместится** в конкретную машину: параметры, квантизация (quantization), RAM;
- прикинуть **время ответа** и понять, почему на CPU болит не длинный ответ, а длинный промпт;
- применять паттерны **OpenAI-совместимого API** и **роутера** (router), чтобы не привязываться к одному провайдеру.

## План

1. Два способа получить LLM в приложение
2. Облачные API: как устроена оплата + считаем деньги
3. Локальные модели: параметры, квантизация, RAM, скорость
4. Критерии выбора: сводная таблица
5. OpenAI-совместимый API и паттерн «роутер»

⏱ **Время:** ~60–75 минут.

> 💡 Этот урок целиком исполняется **без Ollama и без интернета** — здесь только расчёты и модели-на-бумаге. Идеальный урок для поезда.

## 1. Два способа получить LLM в своё приложение

Если вы работали с базами данных, аналогия будет мгновенной: это выбор между **managed-сервисом** (RDS, Cloud SQL — провайдер держит сервер, вы платите за использование) и **self-hosted PostgreSQL** (вы админите сами, зато всё под контролем). С LLM ровно та же развилка:

```text
        ОБЛАЧНЫЙ API                            ЛОКАЛЬНАЯ МОДЕЛЬ

        ваш backend                             ваш backend
            |                                       |
            | HTTPS + API-ключ                      | HTTP (localhost)
            v                                       v
   +----------------------+              +----------------------+
   |  сервер провайдера   |              |  Ollama-процесс      |
   |  (GPU-кластер где-то |              |  на вашей машине /   |
   |   в чужом дата-      |              |  вашем сервере       |
   |   центре)            |              |                      |
   +----------------------+              +----------------------+

   оплата:  за токены                    оплата:  железо + электричество
   данные:  уходят провайдеру            данные:  не покидают ваш контур
   модель:  топовая, но не ваша          модель:  скромнее, но ваша
```

Оба варианта — это просто **HTTP-сервис, который принимает messages и возвращает токены**. Протокол общения вы уже знаете из урока 1.2; меняется только то, кто владеет железом и весами.

### Облачные API: модель как сервис

Провайдер держит модель на своих GPU, вы шлёте запросы по HTTPS с API-ключом. Основные игроки:

| провайдер | модели | особенность |
|---|---|---|
| OpenAI | GPT-серия | де-факто задал стандарт API |
| Anthropic | Claude-серия | сильные модели для кода и длинного контекста |
| Google | Gemini-серия | огромный контекст |
| DeepSeek | DeepSeek-серия | агрессивно низкие цены |
| OpenRouter | агрегатор: сотни моделей за одним API | один ключ — много провайдеров |
| Яндекс / Сбер | YandexGPT, GigaChat | хостинг в РФ — важно для 152-ФЗ |

**Как устроена оплата.** Вы платите за токены (помните урок 1.1?) — отдельно за входные (input, ваш промпт) и выходные (output, ответ модели). Цены указывают за 1 миллион токенов, и — важная деталь — **output-токены обычно в 3–5 раз дороже input**: генерация тяжелее, чем чтение.

**Что вы получаете:** лучшие модели на рынке, старт за 5 минут, никакого ops.

**Что подписываете мелким шрифтом:**
- данные **уходят на сервер провайдера** — для персональных данных нужно читать DPA (data processing agreement) и учитывать GDPR / 152-ФЗ;
- **rate limits** — провайдер ограничивает запросы/токены в минуту, и под нагрузкой вы в них упрётесь;
- модель могут **обновить или отключить «под вами»** (deprecation) — вчерашние промпты могут заработать иначе;
- нужен стабильный интернет, а счёт растёт линейно с трафиком.

In [ ]:
# Единственная зависимость урока — psutil (понадобится ниже,
# чтобы посмотреть характеристики вашей машины).
from importlib.util import find_spec

if find_spec("psutil") is None:
    %pip install -q psutil
else:
    print("Зависимости уже установлены")

### Считаем деньги

Прежде чем спорить «облако или локально», посчитаем. Цены ниже — **ориентиры на август 2026** (порядок величин реальный, но провайдеры меняют прайс регулярно — перед настоящим расчётом сверьтесь с их сайтами). Названия классов моделей: «мини» — дешёвые быстрые модели, «флагман» — самые сильные.

In [ ]:
# Ориентировочные цены (USD за 1 млн токенов), август 2026.
# Перед реальным расчётом сверьтесь с актуальным прайсом провайдера!
PRICES_USD_PER_1M = {
    #  модель           (input, output)
    "gpt-4o-mini":     (0.15,  0.60),   # мини-класс OpenAI
    "gpt-4o":          (2.50, 10.00),   # флагман OpenAI
    "claude-haiku":    (0.80,  4.00),   # быстрый класс Anthropic
    "claude-sonnet":   (3.00, 15.00),   # старший класс Anthropic
    "deepseek-chat":   (0.27,  1.10),   # DeepSeek
}

USD_RUB = 90  # условный курс для прикидок — подставьте актуальный

def request_cost_usd(model: str, input_tokens: int, output_tokens: int) -> float:
    """Стоимость одного запроса к облачной модели, в долларах."""
    price_in, price_out = PRICES_USD_PER_1M[model]
    return (input_tokens / 1_000_000 * price_in
            + output_tokens / 1_000_000 * price_out)

for model in PRICES_USD_PER_1M:
    cost = request_cost_usd(model, input_tokens=63000, output_tokens=57000)
    print(f"Один запрос (63000 in + 57000 out) к {model}: ${cost:.4f} (~{cost * USD_RUB:.2f} руб.)")

Полтора цента за запрос — звучит несерьёзно. Но бэкенд-разработчик знает: всё, что умножается на трафик, перестаёт быть несерьёзным. Возьмём реалистичный сценарий: **бот поддержки** интернет-магазина.

- 2000 запросов в день;
- средний промпт — 1200 токенов (системная инструкция + история диалога + вопрос клиента);
- средний ответ — 300 токенов.

In [ ]:
REQUESTS_PER_DAY = 2000
INPUT_TOKENS = 1200   # системный промпт + история + вопрос
OUTPUT_TOKENS = 300   # ответ бота
DAYS = 30

requests_per_month = REQUESTS_PER_DAY * DAYS
print(f"Нагрузка: {REQUESTS_PER_DAY} запросов/день -> {requests_per_month:,} в месяц\n")

print(f"{'модель':<15} {'$/месяц':>10} {'руб/месяц':>12}")
print("-" * 40)
for model in sorted(PRICES_USD_PER_1M,
                    key=lambda m: request_cost_usd(m, INPUT_TOKENS, OUTPUT_TOKENS)):
    monthly = request_cost_usd(model, INPUT_TOKENS, OUTPUT_TOKENS) * requests_per_month
    print(f"{model:<15} {monthly:>10.2f} {monthly * USD_RUB:>12,.0f}")

Разброс — **в двадцать с лишним раз** между мини-моделью и флагманом на одной и той же нагрузке. Отсюда первое практическое правило: *не гоняйте флагман там, где справится мини* — к маршрутизации задач мы вернёмся в конце урока.

И проверим интуицию: ответ бота в 4 раза короче промпта — значит, и стоит в 4 раза меньше?

In [ ]:
# Какая доля счёта приходится на input, а какая — на output?
print(f"{'модель':<15} {'input, %':>9} {'output, %':>10}")
print("-" * 37)
for model, (price_in, price_out) in PRICES_USD_PER_1M.items():
    cost_in = INPUT_TOKENS / 1e6 * price_in
    cost_out = OUTPUT_TOKENS / 1e6 * price_out
    total = cost_in + cost_out
    print(f"{model:<15} {cost_in / total:>8.0%} {cost_out / total:>9.0%}")

Нет: ответ **в 4 раза короче, но даёт около половины счёта**, потому что output-токен в 4–5 раз дороже. Практический вывод: параметр `max_tokens` из урока 1.2 и просьба «отвечай кратко» в системном промпте — это не только про UX, это ещё и про деньги.

## 2. Локальные модели: модель как процесс

Вторая опция — скачать **открытые веса** (open weights: Llama, Qwen, Mistral и другие семейства) и запустить модель как обычный процесс на своём железе. Инструменты:

| инструмент | что это | когда брать |
|---|---|---|
| **Ollama** | сервер моделей «в один бинарник»: скачивает, запускает, даёт HTTP API | локальная разработка, небольшой прод. **Наш выбор в курсе** (урок 1.5) |
| **llama.cpp** | движок инференса на C++ (Ollama работает поверх него) | тонкая настройка, слабое железо |
| **vLLM** | production-сервер для GPU: батчинг, высокая пропускная способность | серьёзный прод на собственных GPU |
| **LM Studio** | GUI-приложение | посмотреть на модели без консоли |

### Параметры и квантизация: сколько «весит» модель

Размер модели меряют в **параметрах** — тех самых обучаемых весах из урока 1.1. `qwen2.5:3b` — 3 миллиарда параметров, `llama3.3:70b` — 70 миллиардов. Больше параметров → умнее (как правило) → больше памяти и медленнее.

В обучении каждый вес хранится как float16 — 2 байта. Для запуска на обычном железе веса **квантуют** (quantization) — грубят до 8 или даже 4 бит. Аналогия: RAW-фото пережали в JPEG — файл в 4 раза меньше, деталей чуть меньше, но на экране почти не отличить:

| формат | бит на вес | RAM для 8B-модели | качество |
|---|---|---|---|
| F16 | 16 | ~18 ГБ | эталон |
| Q8 | 8 | ~9 ГБ | почти неотличимо |
| **Q4** | 4 | ~4.6 ГБ | лёгкая деградация — **стандарт для локального запуска** |

### Формула для прикидки RAM

```text
RAM (ГБ) ≈ параметры_в_млрд × (бит / 8) × 1.15
```

Множитель 1.15 — накладные расходы. Сверх этого память ест **KV-кэш** — внутренний кэш attention, растущий с длиной контекста: на длинных промптах это ещё гигабайты. Формула — прикидка «влезет / не влезет», а не точный расчёт.

In [ ]:
def model_ram_gb(params_billion: float, bits: int = 4, overhead: float = 1.15) -> float:
    """Грубая оценка RAM под веса модели (без KV-кэша)."""
    return params_billion * bits / 8 * overhead

MODELS = [
    ("qwen2.5:3b",    3),
    ("llama3.1:8b",   8),
    ("qwen2.5:14b",  14),
    ("qwen2.5:32b",  32),
    ("llama3.3:70b", 70),
]

print(f"{'модель':<14} {'Q4':>8} {'Q8':>8} {'F16':>8}")
print("-" * 42)
for name, params in MODELS:
    q4, q8, f16 = (model_ram_gb(params, bits) for bits in (4, 8, 16))
    print(f"{name:<14} {q4:>6.1f}ГБ {q8:>6.1f}ГБ {f16:>6.1f}ГБ")

Теперь главный вопрос локального пути: **что поместится в вашу машину?** Правило: из общей RAM вычтите ~4 ГБ на ОС и остальные процессы — остаток и есть бюджет под модель.

In [ ]:
import os

try:
    import psutil
except ImportError:
    psutil = None
    print("psutil не установлен — выполните первую code-ячейку урока")

if psutil is not None:
    ram_gb = psutil.virtual_memory().total / 2**30
    print(f"Ваша машина: {ram_gb:.1f} ГБ RAM, {os.cpu_count()} логических ядер CPU")

    OS_RESERVE_GB = 4
    budget = ram_gb - OS_RESERVE_GB
    print(f"Бюджет под модель (минус ~{OS_RESERVE_GB} ГБ на ОС): {budget:.1f} ГБ\n")

    for name, params in MODELS:
        need = model_ram_gb(params, bits=4)
        verdict = "поместится" if need <= budget else "НЕ поместится"
        print(f"{name:<14} нужно ~{need:>5.1f} ГБ (Q4) -> {verdict}")

### Скорость: из чего складывается время ответа

Время ответа LLM — это не одно число, а сумма трёх слагаемых:

```text
время ответа = сеть + обработка промпта + генерация ответа
                        (prefill)          (decode)

  обработка промпта:  промпт_токены / скорость_prefill
  генерация:          ответ_токены  / скорость_decode
```

Неочевидный момент: **промпт тоже прогоняется через модель** (этап prefill), прежде чем родится первый токен ответа. На GPU prefill летит (тысячи токенов/с), а вот на CPU он ползёт — и именно это, а не генерация, становится главной болью. Скорости ниже — типичные порядки величин:

In [ ]:
# (prefill ток/с, decode ток/с, сетевая задержка в секундах) — типичные порядки
SETUPS = {
    "облако (флагман)":     (5000, 80, 0.4),
    "локально: CPU, 3B Q4": (60,   15, 0.0),
    "локально: GPU, 8B Q4": (1500, 50, 0.0),
}

def time_to_answer(prompt_tokens: int, output_tokens: int,
                   prefill_speed: float, decode_speed: float, network_s: float) -> float:
    return network_s + prompt_tokens / prefill_speed + output_tokens / decode_speed

PROMPT_TOKENS = 1200
for output in (50, 300):
    print(f"промпт {PROMPT_TOKENS} токенов, ответ {output} токенов:")
    for name, (prefill, decode, net) in SETUPS.items():
        t = time_to_answer(PROMPT_TOKENS, output, prefill, decode, net)
        print(f"  {name:<24} ~{t:>5.1f} с")
    print()

Разбор результатов:

- **Облако** стабильно быстрое: сетевые 300–500 мс — мелочь на фоне остального.
- **CPU** тратит ~20 секунд только на то, чтобы *прочитать* промпт в 1200 токенов. Запомните этот факт: в модуле про RAG мы будем класть в промпт найденные документы, и на CPU каждый лишний килобайт контекста — это секунды ожидания.
- **Потребительский GPU** уже даёт комфортные единицы секунд.

### Качество: честный разговор

Локальная 3B-модель **не равна** облачному флагману — и не надо ждать этого от неё. Но «хуже флагмана» не значит «бесполезна»: для узких задач — классификация обращений, извлечение полей в JSON (урок 1.2!), суммаризация, черновики ответов по шаблону — маленькой модели часто достаточно. А вот сложные многошаговые рассуждения и работа с тонким контекстом — территория больших моделей.

Два правила, которые сэкономят вам месяцы:

1. **Не выбирайте модель по бенчмаркам из интернета.** Их меряют на академических задачах, а не на ваших тикетах. Единственный честный тест — прогнать *свои* примеры (это называется evals, и мы к ним ещё вернёмся).
2. **Сравнивайте модели одного класса.** «Локальная 3B хуже GPT-4o, значит, локальные — мусор» — ошибка масштаба: сравнили мопед с фурой.

В этом курсе мы используем `qwen2.5:3b` через Ollama — потолок ниже облачного, зато бесплатно, приватно и работает на обычном ноутбуке. Для учёбы — идеально.

## 3. Критерии выбора: сводная таблица

| критерий | облачный API | локальная модель |
|---|---|---|
| потолок качества | флагманы — лучшее на рынке | ограничен вашим железом |
| скорость старта | ключ — и через 5 минут работает | установка, подбор модели, железо |
| цена на малой нагрузке | копейки (pay-as-you-go) | железо стоит денег сразу |
| цена на большой нагрузке | растёт линейно с трафиком | фиксированная |
| latency | стабильно низкая | зависит от железа; CPU — медленно |
| приватность | данные уходят провайдеру (читать DPA, GDPR/152-ФЗ) | данные не покидают контур |
| работа offline | нет | да |
| ops-нагрузка | почти нулевая (но: rate limits, деприкейты) | мониторинг, обновления — всё ваше |
| воспроизводимость | модель могут обновить «под вами» | версия зафиксирована вами |

Типичные профили решений:

- **MVP / пет-проект / непредсказуемый трафик** → облако, класс «мини»: быстро и почти бесплатно на старте.
- **Персональные данные, медицина, банк, гостайна** → локально (или облако с хостингом в нужной юрисдикции и подписанным DPA).
- **Стабильно высокая нагрузка на несложных задачах** → локально на своём GPU: фиксированная цена побеждает линейную.
- **Сложные рассуждения, нужен максимум качества** → облачный флагман.

Решение можно сделать и чуть формальнее — взвесить критерии под ваш проект:

In [ ]:
# Оценки 0-10 — субъективные, для ТИПИЧНОГО стартапа без особых требований
# к приватности. Смысл ячейки — способ думать, а не числа. Подставьте свои!
SCORES = {
    #  критерий            вес   облако  локально
    "качество ответов":   (0.25,  9,      6),
    "стоимость на MVP":   (0.20,  9,      7),
    "приватность данных": (0.20,  4,     10),
    "скорость ответа":    (0.15,  8,      5),
    "простота ops":       (0.20,  9,      5),
}

def total_score(option: str) -> float:
    idx = {"облако": 0, "локально": 1}[option]
    return sum(weight * scores[idx] for weight, *scores in SCORES.values())

for option in ("облако", "локально"):
    print(f"{option:<9} итог: {total_score(option):.2f}")

# Попробуйте: поднимите вес приватности до 0.45 (уменьшив другие) -
# каким станет решение для банка?

## 4. OpenAI-совместимый API: страховка от vendor lock-in

Хорошая новость: выбор «облако или локально» **не нужно вшивать в код намертво**. Формат API OpenAI (`POST /v1/chat/completions` с массивом `messages`) стал отраслевым стандартом — его поддерживают почти все: OpenRouter, DeepSeek, vLLM… и **Ollama тоже**.

> 💡 В уроке 1.2 мы ходили в «родной» API Ollama (`/api/chat`) — у неё два интерфейса. Родной даёт больше настроек, совместимый (`/v1/chat/completions`) даёт переносимость.

Смена провайдера сводится к трём строкам конфига: `base_url`, API-ключ, имя модели. Тело запроса не меняется вообще:

In [ ]:
import json

MESSAGES = [
    {"role": "system", "content": "Ты — ассистент службы поддержки магазина 'Ромашка'."},
    {"role": "user", "content": "Как мне сбросить пароль?"},
]

def build_request(provider: str):
    """Один и тот же запрос для разных провайдеров: меняется только конфиг."""
    payload = {"model": None, "messages": MESSAGES,
               "temperature": 0.3, "max_tokens": 200}
    if provider == "openai":
        url = "https://api.openai.com/v1/chat/completions"
        headers = {"Authorization": "Bearer <OPENAI_API_KEY>"}
        payload["model"] = "gpt-4o-mini"
    elif provider == "openrouter":
        url = "https://openrouter.ai/api/v1/chat/completions"
        headers = {"Authorization": "Bearer <OPENROUTER_API_KEY>"}
        payload["model"] = "deepseek/deepseek-chat"
    elif provider == "ollama":
        url = "http://localhost:11434/v1/chat/completions"
        headers = {}  # локально ключ не нужен
        payload["model"] = "qwen2.5:3b"
    else:
        raise ValueError(f"неизвестный провайдер: {provider}")
    return url, headers, payload

for provider in ("openai", "openrouter", "ollama"):
    url, headers, payload = build_request(provider)
    print(f"=== {provider} ===")
    print("POST   ", url)
    print("headers", headers)
    print("payload", json.dumps(payload, ensure_ascii=False)[:100] + "...")
    print()

Практический вывод для архитектуры: **выносите `base_url`, ключ и имя модели в конфигурацию** (переменные окружения, settings), как вы делаете с DSN базы данных. Тогда «переехать с OpenAI на Ollama» — это правка `.env`, а не рефакторинг.

## 5. Гибрид: паттерн «роутер»

Выбор не обязан быть бинарным. Зрелые системы часто **маршрутизируют** запросы: часть трафика — в локальную модель, часть — в облако, по правилам:

```text
                     +----------+
     запрос  ----->  |  роутер  |
                     +----------+
                      /    |    \
        перс. данные/  сложная   \ всё
          offline   |   задача    | остальное
                    v      v      v
                 local   cloud   cloud
                        flagship  mini
```

Роутер — это не магия, а обычная функция с бизнес-правилами:

In [ ]:
def route(*, contains_pii: bool, needs_top_quality: bool = False,
          needs_offline: bool = False) -> str:
    """Решает, куда отправить запрос. Порядок правил важен!"""
    if needs_offline:
        return "local"           # без сети выбора нет
    if contains_pii:
        return "local"           # персональные данные не покидают контур
    if needs_top_quality:
        return "cloud-flagship"  # сложное рассуждение — флагману
    return "cloud-mini"          # всё остальное — дёшево и быстро

# мини-тесты: правила приватности сильнее правил качества
assert route(contains_pii=True, needs_top_quality=True) == "local"
assert route(contains_pii=False, needs_top_quality=True) == "cloud-flagship"
assert route(contains_pii=False) == "cloud-mini"
assert route(contains_pii=False, needs_offline=True) == "local"
print("Роутер работает: все проверки пройдены")

## ⚠️ Частые ошибки

1. **Сравнивать локальную 3B с облачным флагманом** и делать вывод «локальные модели — мусор». Это разные весовые категории; сравнивайте с моделью того же класса и на *своих* задачах.
2. **Считать стоимость только по input-токенам.** Output-токены в 3–5 раз дороже; в наших расчётах ответ в 4 раза короче промпта давал половину счёта.
3. **Верить, что «локально = бесплатно».** Железо, электричество, а главное — ваше время на установку, мониторинг и обновления. Ops-часы тоже стоят денег.
4. **Отправлять персональные данные в облако «по умолчанию»,** не проверив DPA провайдера и требования GDPR / 152-ФЗ. Это риск не технический, а юридический — из тех, что закрывают проекты.
5. **Забывать про KV-кэш и prefill.** RAM считают только по весам, а длинный контекст добавляет гигабайты; latency считают только по ответу, а на CPU промпт в 1200 токенов — это ~20 секунд *до первого токена*.
6. **Зашивать провайдера в код.** URL, ключ и имя модели — это конфигурация. Иначе смена провайдера превращается в рефакторинг всего проекта.

## 🏋️ Упражнения

### Упражнение 1 — точка безубыточности 💰

Аренда сервера с GPU стоит **35 000 руб/мес**. При какой нагрузке (запросов/день, профиль 1200 in + 300 out) месячный счёт за облако сравняется с арендой? Посчитайте для **каждой** модели из `PRICES_USD_PER_1M`. Что заметите, сравнив «мини»-модели и флагманы?

### Упражнение 2 — подбор модели под железо 🧮

Напишите функцию `fits(params_billion, ram_gb, bits=4, os_reserve_gb=4.0) -> bool`, которая отвечает, поместится ли модель в машину с данным объёмом RAM (с резервом под ОС). Проверьте все модели из `MODELS` для машины с 16 ГБ — в Q4 и Q8.

### Упражнение 3 — роутер v2 🔀

Расширьте `route()` двумя параметрами: `local_available: bool` (локальная модель может быть недоступна — упала, обновляется) и `monthly_budget_left_usd: float` (остаток облачного бюджета). Новые правила:

- запрос с персональными данными при недоступной локальной модели **нельзя** отправлять в облако — верните `"queue"` (отложить);
- при исчерпанном бюджете (`<= 0`) весь неприватный трафик идёт в локальную модель, а если она недоступна — в `"queue"`.

Покройте новые ветки `assert`-ами.

## 📝 Мини-квиз

**1. За что вы платите облачному LLM-провайдеру?**

<details><summary>Ответ</summary>

За **токены**: отдельно за входные (промпт) и выходные (ответ), по разным ценам за миллион. Output-токены обычно в 3–5 раз дороже input.

</details>

**2. Почему нельзя оценивать стоимость запроса только по длине промпта?**

<details><summary>Ответ</summary>

Потому что output-токены заметно дороже. В нашем сценарии ответ был в 4 раза короче промпта, но давал ~половину счёта.

</details>

**3. Что делает квантизация Q4 и чем вы за это платите?**

<details><summary>Ответ</summary>

Хранит веса модели в 4 битах вместо 16 → примерно **в 4 раза меньше RAM** (и быстрее чтение весов). Плата — небольшая потеря качества ответов. Для локального запуска Q4 — стандартный выбор.

</details>

**4. Почему на CPU медленно даже «до первого токена ответа»?**

<details><summary>Ответ</summary>

Из-за этапа **prefill**: промпт тоже прогоняется через модель. На CPU это десятки токенов в секунду — промпт в 1200 токенов обрабатывается ~20 секунд ещё до начала генерации.

</details>

**5. Вы переезжаете с OpenAI на Ollama через OpenAI-совместимый API. Что придётся поменять в коде?**

<details><summary>Ответ</summary>

Только конфигурацию: `base_url` (на `http://localhost:11434/v1`), API-ключ (не нужен) и имя модели. Формат запроса и ответа остаётся тем же — поэтому эти три вещи и стоит держать в конфиге, а не в коде.

</details>

## ✅ Решения упражнений

Сначала попробуйте сами! Подглядывать сюда — только после честной попытки.

### Решение 1 — точка безубыточности

In [ ]:
GPU_SERVER_RUB = 35_000
threshold_usd = GPU_SERVER_RUB / USD_RUB

print(f"Порог: ${threshold_usd:.0f}/мес (аренда GPU-сервера {GPU_SERVER_RUB:,} руб)\n")
print(f"{'модель':<15} {'break-even, запросов/день':>27}")
print("-" * 44)
for model in sorted(PRICES_USD_PER_1M,
                    key=lambda m: request_cost_usd(m, 1200, 300)):
    per_request = request_cost_usd(model, 1200, 300)
    req_per_day = threshold_usd / per_request / 30
    print(f"{model:<15} {req_per_day:>22,.0f}")

Вывод из чисел: у флагманов точка безубыточности наступает уже при ~1.5–2 тысячах запросов в день — заметная, но достижимая нагрузка. У мини-моделей — при десятках тысяч запросов в день: облако для них дешевле почти всегда. Но помните про честность сравнения: на арендованном GPU вы запустите модель классом ниже флагмана, так что считать break-even имеет смысл только для задач, где локального качества достаточно.

### Решение 2 — подбор модели под железо

In [ ]:
def fits(params_billion: float, ram_gb: float,
         bits: int = 4, os_reserve_gb: float = 4.0) -> bool:
    """Поместится ли модель в машину с ram_gb памяти (с резервом под ОС)."""
    return model_ram_gb(params_billion, bits) <= ram_gb - os_reserve_gb

RAM = 16
print(f"Машина с {RAM} ГБ RAM (резерв под ОС — 4 ГБ):\n")
print(f"{'модель':<14} {'Q4':>5} {'Q8':>5}")
print("-" * 26)
for name, params in MODELS:
    q4 = "да" if fits(params, RAM, bits=4) else "нет"
    q8 = "да" if fits(params, RAM, bits=8) else "нет"
    print(f"{name:<14} {q4:>5} {q8:>5}")

На 16 ГБ в Q4 комфортно живут модели до 14B, а вот 32B уже не влезает; в Q8 потолок — 8B. Именно поэтому «какая у вас RAM» — первый вопрос при выборе локальной модели.

### Решение 3 — роутер v2

In [ ]:
def route_v2(*, contains_pii: bool, needs_top_quality: bool = False,
             needs_offline: bool = False, local_available: bool = True,
             monthly_budget_left_usd: float = 100.0) -> str:
    """Роутер с учётом доступности локальной модели и облачного бюджета."""
    if needs_offline or contains_pii:
        # приватному и offline-трафику в облако нельзя ни при каких условиях
        return "local" if local_available else "queue"
    if monthly_budget_left_usd <= 0:
        return "local" if local_available else "queue"
    if needs_top_quality:
        return "cloud-flagship"
    return "cloud-mini"

# приватность важнее всего: облако не рассматривается даже при аварии local
assert route_v2(contains_pii=True) == "local"
assert route_v2(contains_pii=True, local_available=False) == "queue"

# бюджет исчерпан: неприватный трафик уходит в local, при аварии - в очередь
assert route_v2(contains_pii=False, monthly_budget_left_usd=0) == "local"
assert route_v2(contains_pii=False, monthly_budget_left_usd=0,
                local_available=False) == "queue"

# обычные сценарии не сломались
assert route_v2(contains_pii=False, needs_top_quality=True) == "cloud-flagship"
assert route_v2(contains_pii=False) == "cloud-mini"

print("Роутер v2 работает: все проверки пройдены")

Обратите внимание на порядок правил: сначала **безопасность** (приватное — никогда в облако), потом **деньги** (бюджет), и только потом **качество**. Поменяете порядок — и при исчерпанном бюджете персональные данные поедут во флагман. В реальном сервисе такой роутер обрастает логированием и метриками, но скелет — ровно этот.

## 🎓 Итоги

- **Облачный API** — модель как сервис: лучшие модели, нулевой ops, оплата за токены (output в 3–5 раз дороже input), но данные уходят провайдеру, а счёт растёт линейно с трафиком.
- **Локальная модель** — модель как процесс: данные не покидают контур, цена фиксированная, но потолок качества задаёт ваше железо. RAM прикидываем как `параметры × бит/8 × 1.15`, стандарт квантизации — Q4.
- Время ответа = сеть + **prefill** + генерация; на CPU главная боль — обработка длинного промпта, что особенно важно для RAG.
- Точку безубыточности **считают, а не угадывают** — у флагманов она наступает при тысячах запросов в день, у мини-моделей почти никогда.
- **OpenAI-совместимый API** превращает выбор провайдера в конфигурацию, а паттерн **«роутер»** позволяет не выбирать вовсе — маршрутизировать по правилам: приватность → деньги → качество.

**Дальше — урок 1.4 «Первый запрос к LLM»:** хватит считать на бумаге, начинаем писать рабочий код против API.

## 📚 Что почитать

- [Ollama: OpenAI compatibility](https://ollama.com/blog/openai-compatibility) — совместимый API, который мы обсуждали
- [Прайс OpenAI](https://openai.com/api/pricing/) и [прайс Anthropic](https://www.anthropic.com/pricing) — актуальные цены за токены
- [OpenRouter](https://openrouter.ai/models) — агрегатор моделей: удобно сравнивать цены в одном месте
- [llama.cpp](https://github.com/ggml-org/llama.cpp) — движок локального инференса; в README много деталей про квантизацию
- [vLLM](https://docs.vllm.ai/) — если однажды понадобится серьёзный прод на своих GPU